In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.load import list_children, load_child
from src.features import window_features

parts = []
for _, k in pd.DataFrame(list_children()).iterrows():
    f = window_features(load_child(k.path))
    f["child_id"], f["label"] = k.child_id, k.label
    parts.append(f)
win = pd.concat(parts, ignore_index=True)

X, y, g = win.drop(columns=["child_id", "label"]), win.label, win.child_id
print(win.shape)
print(win.groupby("label").size())

(4173, 78)
label
0    1843
1    2330
dtype: int64


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, cross_validate

model = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
splits = {
    "PAPER WAY (pieces shuffled)": StratifiedKFold(5, shuffle=True, random_state=42),
    "HONEST (split by child)":     StratifiedGroupKFold(5, shuffle=True, random_state=42),
}
for name, cv in splits.items():
    r = cross_validate(model, X, y, groups=g, cv=cv, scoring=["accuracy", "roc_auc"])
    print(f"{name:28s} accuracy {r['test_accuracy'].mean():.2f}   AUC {r['test_roc_auc'].mean():.2f}")

c:\Users\Rajesh Sampathkumar\miniconda3\envs\medimg\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


PAPER WAY (pieces shuffled)  accuracy 0.79   AUC 0.88
HONEST (split by child)      accuracy 0.63   AUC 0.64


In [3]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

honest = StratifiedGroupKFold(5, shuffle=True, random_state=42)
p = cross_val_predict(model, X, y, groups=g, cv=honest, method="predict_proba")[:, 1]
per_child = pd.DataFrame({"child_id": g, "label": y, "p": p}).groupby("child_id").mean()
print("Honest, per-child AUC:", round(roc_auc_score(per_child.label, per_child.p), 2))

Honest, per-child AUC: 0.7


In [4]:
from sklearn.model_selection import cross_val_score
who = cross_val_score(model, X, g, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="accuracy")
print(f"Who-is-this accuracy: {who.mean():.2f}  (guessing = 0.008)")

Who-is-this accuracy: 0.70  (guessing = 0.008)
